In [5]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Configuration
RANDOM_STATE = 42
RAW_CSV_PATH = "raw_loan_dataset.csv"

print("--- Step 1: Processing & Cleaning Data ---")
# 1. Soo rar xogta cayriin
df = pd.read_csv(RAW_CSV_PATH)

# 2. Beddel eberada khaldan (zeros) oo ka dhig NaN, ka dibna ku buuxi median-ka
continuous_cols = ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount']
for col in continuous_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

# 3. Dib u xisaabi LoanToIncomeRatio si ay u noqoto mid sax ah
df['LoanToIncomeRatio'] = df['LoanAmount'] / df['Income']

# 4. Hubi in flag-yadu ay yihiin integer
flag_cols = ['HasCollateral', 'PreviousDefaults', 'Approved']
df[flag_cols] = df[flag_cols].astype(int)

print(f"Data processed successfully. Shape: {df.shape}")

print("\n--- Step 2: Splitting Data ---")
X = df.drop(columns=["Approved"])
y = df["Approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")

print("\n--- Step 3: Training Machine Learning Models ---")
# Halkan waxaan ku tababaraynaa saddexda moodel
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
xgb_model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss")

log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
print("All models trained successfully!")

print("\n--- Step 4: Evaluating Performance ---")
# Functions loo isticmaalayo in lagu daabaco metrics-ka iyo confusion matrix-ka
def print_clf_metrics(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"\n[{name} Performance]")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall   : {rec:.3f}")
    print(f"  F1-Score : {f1:.3f}")

def print_confmat(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    cm_df = pd.DataFrame(
        cm,
        index=["Actual: Approved (1)", "Actual: Rejected (0)"],
        columns=["Pred: Approved (1)", "Pred: Rejected (0)"],
    )
    print(f"\n{name} – Confusion Matrix:\n{cm_df}")

# Logistic Regression Evaluation
log_reg_preds = log_reg.predict(X_test)
print_clf_metrics("Logistic Regression", y_test, log_reg_preds)
print_confmat("Logistic Regression", y_test, log_reg_preds)

# Random Forest Evaluation
rf_preds = rf.predict(X_test)
print_clf_metrics("Random Forest", y_test, rf_preds)
print_confmat("Random Forest", y_test, rf_preds)

# XGBoost Evaluation
xgb_preds = xgb_model.predict(X_test)
print_clf_metrics("XGBoost", y_test, xgb_preds)
print_confmat("XGBoost", y_test, xgb_preds)

print("\n--- Step 5: Live Test Case (Row 12) ---")
def label_str(v):
    return "Approved (1)" if v == 1 else "Rejected (0)"

i = 12
x_one_df = X_test.iloc[[i]]
y_true_single = y_test.iloc[i]

print(f"Actual Label      : {label_str(y_true_single)}")
print(f"Logistic Reg Pred : {label_str(int(log_reg.predict(x_one_df)[0]))}")
print(f"Random Forest Pred: {label_str(int(rf.predict(x_one_df)[0]))}")
print(f"XGBoost Pred      : {label_str(int(xgb_model.predict(x_one_df)[0]))}")

--- Step 1: Processing & Cleaning Data ---


FileNotFoundError: [Errno 2] No such file or directory: 'raw_loan_dataset.csv'